## Silver Layer — Data Cleaning
Filtering to delivered orders only (removing cancelled, unavailable)

In [0]:
BRONZE_PATH = "/Volumes/workspace/default/raw_data/bronze"
SILVER_PATH = "/Volumes/workspace/default/raw_data/silver"

# Reading all Bromze Delta Tables
orders      = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")
customers   = spark.read.format("delta").load(f"{BRONZE_PATH}/customers")
order_items = spark.read.format("delta").load(f"{BRONZE_PATH}/order_items")
payments    = spark.read.format("delta").load(f"{BRONZE_PATH}/payments")
reviews     = spark.read.format("delta").load(f"{BRONZE_PATH}/reviews")
products    = spark.read.format("delta").load(f"{BRONZE_PATH}/products")
sellers     = spark.read.format("delta").load(f"{BRONZE_PATH}/sellers")
category    = spark.read.format("delta").load(f"{BRONZE_PATH}/category_translation")

print(f"Orders count: {orders.count()}")
print(f"Customers count: {customers.count()}")

In [0]:
print("ORDERS SCHEMA")
orders.printSchema()

print("\nORDERS STATUS DISTRIBUTION")
orders.groupBy("order_status").count() \
    .orderBy("count", ascending=False).show()

print("\nDATE RANGE OF DATASET")
orders.selectExpr(
    "min(order_purchase_timestamp) as earliest_order",
    "max(order_purchase_timestamp) as latest_order"
).show()

In [0]:
from pyspark.sql.functions import col, to_timestamp, when

# Casting timestamp columns properly
orders_clean = orders \
    .withColumn("order_purchase_timestamp", 
                to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_delivered_customer_date", 
                to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", 
                to_timestamp(col("order_estimated_delivery_date"))) \
    .filter(col("order_status") == "delivered") \
    .filter(col("order_purchase_timestamp").isNotNull()) \
    .dropDuplicates(["order_id"])

print(f"Orders cleaned: {orders_clean.count()} delivered orders")

In [0]:
from pyspark.sql.functions import isnan, when, count, col

null_counts = orders_clean.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in orders_clean.columns
])
print("NULL COUNTS AFTER CLEANING:")
null_counts.show()

In [0]:
from pyspark.sql.functions import sum as spark_sum, avg, count, max as spark_max

# Aggregate payments per order
payments_agg = payments.groupBy("order_id").agg(
    spark_sum("payment_value").alias("total_payment_value"),
    avg("payment_installments").alias("avg_installments")
)

# Aggregate reviews per order
reviews_agg = reviews.groupBy("order_id").agg(
    avg("review_score").alias("avg_review_score")
)

# Aggregate order items per order
items_agg = order_items.groupBy("order_id").agg(
    spark_sum("price").alias("total_items_price"),
    spark_sum("freight_value").alias("total_freight"),
    count("order_item_id").alias("items_count")
)

# Master join
orders_master = orders_clean \
    .join(customers, "customer_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews_agg, "order_id", "left") \
    .join(items_agg, "order_id", "left")

print(f"Master orders table built: {orders_master.count()} rows")
print(f"Columns: {len(orders_master.columns)}")

In [0]:
from pyspark.sql.functions import datediff, lit
import pyspark.sql.functions as F

# Reference date = most recent order date in dataset
max_date = orders_master.agg(
    spark_max("order_purchase_timestamp")).collect()[0][0]

# Build RFM per customer
rfm = orders_master.groupBy("customer_unique_id").agg(
    datediff(lit(max_date),
             spark_max("order_purchase_timestamp")).alias("recency_days"),
    count("order_id").alias("frequency"),
    spark_sum("total_payment_value").alias("monetary_value"),
    avg("avg_review_score").alias("avg_review_score"),
    avg("items_count").alias("avg_items_per_order"),
    avg("total_freight").alias("avg_freight_value"),
    spark_max("customer_state").alias("customer_state"),
    spark_max("customer_city").alias("customer_city")
)

print(f"RFM table built: {rfm.count()} unique customers")

In [0]:
print("RFM SUMMARY STATISTICS")
rfm.select(
    "recency_days", "frequency", 
    "monetary_value", "avg_items_per_order"
).describe().show()

print("\nTOP 10 STATES BY CUSTOMER COUNT")
rfm.groupBy("customer_state") \
   .count() \
   .orderBy("count", ascending=False) \
   .show(10)

In [0]:
# churn — customer hasn't ordered in last 180 days
# Recreate rfm without the problematic avg_review_score column
rfm_clean = orders_master.groupBy("customer_unique_id").agg(
    datediff(lit(max_date),
             spark_max("order_purchase_timestamp")).alias("recency_days"),
    count("order_id").alias("frequency"),
    spark_sum("total_payment_value").alias("monetary_value"),
    avg("items_count").alias("avg_items_per_order"),
    avg("total_freight").alias("avg_freight_value"),
    spark_max("customer_state").alias("customer_state"),
    spark_max("customer_city").alias("customer_city")
)

rfm_labeled = rfm_clean.withColumn("is_churned",
    when(col("recency_days") > 180, 1).otherwise(0)
)

churn_rate = rfm_labeled.filter(
    col("is_churned") == 1).count() / rfm_labeled.count() * 100
print(f"Churn rate in dataset: {churn_rate:.1f}%")

# Write to Silver as Delta
rfm_labeled.write.format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/customer_rfm_features")

print(f"Silver RFM feature table written")
print(f"Location: {SILVER_PATH}/customer_rfm_features")